# Gemma 2 2B Real-Model V2 Run

This notebook mirrors `docs/COLAB_GEMMA.md`. Run it in a GPU Colab runtime. Do not claim Gemma results unless the cells complete and artifacts are saved.

In [ ]:
!git clone <REPO_URL> neural-native-software
%cd neural-native-software
!python -m pip install --upgrade pip
!python -m pip install -e ".[dev,llm]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")

In [ ]:
!python scripts/generate_dataset_v2.py --output data/intent_dataset_v2.csv --seed 42 --examples-per-class 60 --abstain-examples 180 --strict
!python scripts/audit_dataset.py --dataset data/intent_dataset_v2.csv --output-json artifacts/dataset_v2_audit.json --output-csv artifacts/dataset_v2_near_duplicates.csv

In [ ]:
MODEL_ID = "google/gemma-2-2b-it"
!python scripts/extract_features.py --dataset data/intent_dataset_v2.csv --model-id {MODEL_ID} --batch-size 2 --max-length 160 --output artifacts/features_gemma2_2b_v2_pre_lm_head.npz

In [ ]:
!python scripts/train_probe.py --features artifacts/features_gemma2_2b_v2_pre_lm_head.npz --output artifacts/probe_gemma2_2b_v2.joblib --metrics artifacts/metrics_gemma2_2b_v2.json --confusion-matrix artifacts/confusion_matrix_gemma2_2b_v2.csv --thresholds artifacts/thresholds_gemma2_2b_v2.json --random-state 42

In [ ]:
!python scripts/evaluate_hard_eval.py --model-id {MODEL_ID} --probe artifacts/probe_gemma2_2b_v2.joblib --dataset data/hard_eval.csv --output-metrics artifacts/hard_eval_metrics_gemma2_2b_v2.json --output-predictions artifacts/hard_eval_predictions_gemma2_2b_v2.csv --output-confusion artifacts/hard_eval_confusion_matrix_gemma2_2b_v2.csv --features-output artifacts/hard_eval_features_gemma2_2b_v2_pre_lm_head.npz --batch-size 2

In [ ]:
!python scripts/calibrate_router.py --probe artifacts/probe_gemma2_2b_v2.joblib --features artifacts/features_gemma2_2b_v2_pre_lm_head.npz --output-calibration artifacts/router_calibration_gemma2_2b_v2.json --output-thresholds artifacts/router_thresholds_calibrated_gemma2_2b_v2.json
!python scripts/evaluate_hard_eval.py --model-id {MODEL_ID} --probe artifacts/probe_gemma2_2b_v2.joblib --dataset data/hard_eval.csv --output-metrics artifacts/hard_eval_calibrated_gemma2_2b_v2.json --output-predictions artifacts/hard_eval_predictions_calibrated_gemma2_2b_v2.csv --output-confusion artifacts/hard_eval_confusion_matrix_calibrated_gemma2_2b_v2.csv --thresholds-json artifacts/router_thresholds_calibrated_gemma2_2b_v2.json --batch-size 2

In [ ]:
!python scripts/run_scripted_demo.py --model-id {MODEL_ID} --probe artifacts/probe_gemma2_2b_v2.joblib --thresholds-json artifacts/router_thresholds_calibrated_gemma2_2b_v2.json --example-set v2 --output artifacts/example_routes_gemma2_2b_v2.jsonl --summary-output docs/DEMO_RESULTS_GEMMA2_2B_V2.md --batch-size 6